split into CVs

add correlation-based selection

1) load and analyze variance of descriptors in all the datasets
2) remove descriptors with low variance
4) add gaussian processes regression and bayesian linear regression 
5) add hyperparameter optimization

when done with all the datasets, analyze which descriptors are most important for each property
select ~20

debug round 5

- optimize parallel script in terms of logging and run on structures

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from pathlib import Path
import scipy.stats as stats
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import glob
import os
import itertools
pd.set_option('display.max_rows', None)
warnings.filterwarnings('ignore')
from collections import Counter
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import spearmanr

from utils.load_results_to_dataframe import load_json_results

# Utils

In [18]:
def load_and_merge(exp_csv_path: Path, results_dir_path: Path, base: str):
    """Load experimental CSV + descriptor JSONs, then inner-join on `name`."""
    df_results = load_json_results(results_dir_path)
    df_results["base"] = base

    exp_df = pd.read_csv(exp_csv_path)

    if "name" not in exp_df.columns:
        raise ValueError(f"Experimental CSV {exp_csv_path} must contain a 'name' column.")
    if "name" not in df_results.columns:
        raise ValueError(
            f"Results loaded from {results_dir_path} are missing a 'name' column for merging."
        )

    try:
        exp_df["name"] = exp_df["name"].astype(int)
    except:
        pass
    try:
        df_results["name"] = df_results["name"].astype(int)
    except:
        pass
    
    exp_df["name"] = exp_df["name"].astype(str)
    df_results["name"] = df_results["name"].astype(str)


    merged_df = exp_df.merge(df_results, on="name", how="inner")
    print(
        f"Merged experimental {Path(exp_csv_path).name} with results '{base}': {len(merged_df)} rows"
    )
    return merged_df


In [19]:
def remove_low_variance_features(
    X: pd.DataFrame,
    feature_cols: list,
    relative_std_threshold: float = 0.01,
    epsilon: float = 1e-8,
):
    """Low-variance feature removal using relative std.

    relative_std = std(feature) / (|mean(feature)| + epsilon)

    Remove features when: relative_std < relative_std_threshold
    (default 0.01, i.e. 1% safe cutoff).

    Parameters
    ----------
    X:
        DataFrame containing the candidate feature columns (typically the training fold only).
    feature_cols:
        List of candidate feature column names.
    relative_std_threshold:
        Cutoff for dropping low-variance features.
    epsilon:
        Small constant to avoid division by zero when mean ~ 0.

    Returns
    -------
    kept_features : list[str]
    removed_features : list[str]
    relative_std : pd.Series
        relative_std indexed by feature name.
    """
    def _column_as_series(df: pd.DataFrame, col: str) -> pd.Series:
        s = df[col]
        if isinstance(s, pd.DataFrame):
            return s.iloc[:, 0]
        return s

    if feature_cols is None:
        feature_cols = []
    feature_cols = list(feature_cols)

    if len(feature_cols) == 0:
        empty = pd.Series(dtype=float)
        return [], [], empty

    feature_cols = list(dict.fromkeys(feature_cols))

    coerced = {
        c: pd.to_numeric(_column_as_series(X, c), errors="coerce") for c in feature_cols
    }
    X_feat = pd.DataFrame(coerced)

    mean = X_feat.mean(axis=0, skipna=True)
    std = X_feat.std(axis=0, ddof=0, skipna=True)
    relative_std = std / (mean.abs() + epsilon)

    keep_mask = np.isfinite(relative_std) & (relative_std >= relative_std_threshold)
    kept_features = relative_std.index[keep_mask].tolist()
    removed_features = relative_std.index[~keep_mask].tolist()

    return kept_features, removed_features, relative_std

In [20]:
def _bh_adjust(pvals):
    """Benjamini-Hochberg FDR adjustment. pvals: array-like. NaNs are preserved (not used in adjustment)."""
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan)
    valid = np.isfinite(p)
    if not np.any(valid):
        return p
    p_valid = p[valid]
    n = len(p_valid)
    order = np.argsort(p_valid)
    p_sorted = p_valid[order]
    ratios = n * p_sorted / np.arange(1, n + 1)
    adj_sorted = np.minimum(1, np.minimum.accumulate(ratios[::-1])[::-1])
    rank_of_original = np.argsort(order)
    out[valid] = adj_sorted[rank_of_original]
    return out

def calculate_correlations_and_plot(
    merged_df,
    target_col,
    p_threshold=0.05,
    fdr_alpha=0.05,
    use_fdr=True,
    normalize=False,
    make_plots=False,
    correlation_threshold=None, 
):
    """Compute Spearman correlations for one or multiple targets.

    Parameters
    ----------
    target_col : str or list[str]
        Target column name(s).

    Returns
    -------
    corr_df_by_target : pd.DataFrame or dict
        For a single target: the correlation dataframe.
        For multiple targets: {target_col: corr_df}.

    significant_by_target : dict
        {target_col: [(feature, spearman_r), ...]}
    """

    if isinstance(target_col, str):
        target_cols = [target_col]
        scalar_target = True
    else:
        # Accept list/tuple/set/np.ndarray/pd.Series.
        target_cols = list(target_col)
        scalar_target = False

    if len(target_cols) == 0:
        raise ValueError("target_col must be a non-empty string or iterable of strings")

    exclude_cols = ['antibody_id', 'residue_number', 'n_total_rows', 'n_filtered_rows', 'n_beta_sheet_rows', 'n_exposed_rows']
    id_cols = {'structure_id', 'base', 'heavy', 'light', 'dataset', 'name', 'antibody_name'}

    corr_df_by_target = {}
    significant_by_target = {}

    for tcol in target_cols:
        merged_df_t = merged_df.copy()
        merged_df_t[tcol] = pd.to_numeric(merged_df_t[tcol], errors="coerce")

        n_before = len(merged_df_t)
        merged_df_t = merged_df_t.dropna(subset=[tcol])
        n_after = len(merged_df_t)
        if n_before > n_after:
            print(f"Dropped {n_before - n_after} rows with NaN/invalid target '{tcol}' (using {n_after} for correlations).")

        numeric_cols = merged_df_t.select_dtypes(include=[np.number]).columns.tolist()
        if len(numeric_cols) > 0:
            feature_cols = [
                col
                for col in numeric_cols
                if col not in exclude_cols
                and col not in id_cols
                and col not in target_cols
                and not str(col).startswith("target")
            ]
        else:
            feature_cols = [
                col
                for col in merged_df_t.columns
                if col not in exclude_cols
                and col not in id_cols
                and col not in target_cols
                and not str(col).startswith("target")
            ]

        correlations = []
        for col in feature_cols:
            data = merged_df_t[[tcol, col]].copy()
            data[tcol] = pd.to_numeric(data[tcol], errors='coerce')
            data[col] = pd.to_numeric(data[col], errors='coerce')
            data = data.dropna()
            if len(data) < 3:
                continue

            if normalize:
                target_min = data[tcol].min()
                target_max = data[tcol].max()
                target_range = target_max - target_min
                if target_range > 0:
                    target_norm = (data[tcol] - target_min) / target_range
                else:
                    target_norm = data[tcol]

                feature_min = data[col].min()
                feature_max = data[col].max()
                feature_range = feature_max - feature_min
                if feature_range > 0:
                    feature_norm = (data[col] - feature_min) / feature_range
                else:
                    feature_norm = data[col]

                spearman_r, spearman_p = spearmanr(target_norm, feature_norm)
            else:
                spearman_r, spearman_p = spearmanr(data[tcol], data[col])

            correlations.append({
                'feature': col,
                'spearman_r': spearman_r,
                'spearman_p': spearman_p,
                'n_samples': len(data)
            })

        corr_df = pd.DataFrame(correlations)
        if corr_df.empty:
            corr_df = pd.DataFrame(columns=['feature', 'spearman_r', 'spearman_p', 'spearman_p_adj', 'n_samples'])
            significant = corr_df.copy()
        else:
            if use_fdr:
                corr_df['spearman_p_adj'] = _bh_adjust(corr_df['spearman_p'].values)
                significant = corr_df[corr_df['spearman_p_adj'] < fdr_alpha].copy()
            else:
                corr_df['spearman_p_adj'] = corr_df['spearman_p'].values
                significant = corr_df[corr_df['spearman_p'] < p_threshold].copy()

            # NEW: apply correlation magnitude threshold
            if correlation_threshold is not None:
                significant = significant[
                    significant["spearman_r"].abs() >= correlation_threshold
                ].copy()

        print(f"[target={tcol}] Total features tested: {len(corr_df)}")
        print(f"[target={tcol}] Significant (raw p < {p_threshold}): {(corr_df['spearman_p'] < p_threshold).sum()}")
        if use_fdr:
            print(f"[target={tcol}] Significant after FDR correction (adj p < {fdr_alpha}): {len(significant)}")
        else:
            print(f"[target={tcol}] Significant (no FDR): {len(significant)}")
        if correlation_threshold is not None:
            print(
                f"[target={tcol}] After applying |rho| >= {correlation_threshold}: {len(significant)}"
            )
        if normalize:
            print(f"[target={tcol}] Note: Features were min-max normalized before correlation calculation")
        print(
            f"[target={tcol}] Top correlations by absolute Spearman r ({'FDR-significant only' if use_fdr else 'raw p < ' + str(p_threshold)}):"
        )

        if len(significant) > 0:
            sig = significant.copy()
            sig["spearman_r"] = pd.to_numeric(sig["spearman_r"], errors="coerce")
            sig = sig.dropna(subset=["spearman_r"])
            if len(sig) > 0:
                cols = ["feature", "spearman_r", "spearman_p", "spearman_p_adj"]
                print(sig.nlargest(10, "spearman_r", keep="all")[[c for c in cols if c in sig.columns]])
            else:
                print("(none)")
        else:
            print("(no significant correlations)")

        if make_plots and len(significant) > 0:
            n_plots = len(significant)
            n_cols = 3
            n_rows = (n_plots + n_cols - 1) // n_cols

            fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
            axes = np.atleast_1d(axes).flatten()

            for plot_idx, (idx, row) in enumerate(significant.iterrows()):
                ax = axes[plot_idx]
                data = merged_df_t[[tcol, row['feature']]].copy()
                data[tcol] = pd.to_numeric(data[tcol], errors='coerce')
                data[row['feature']] = pd.to_numeric(data[row['feature']], errors='coerce')
                data = data.dropna()

                if normalize:
                    target_min = data[tcol].min()
                    target_max = data[tcol].max()
                    target_range = target_max - target_min
                    if target_range > 0:
                        target_norm = (data[tcol] - target_min) / target_range
                    else:
                        target_norm = data[tcol]

                    feature_min = data[row['feature']].min()
                    feature_max = data[row['feature']].max()
                    feature_range = feature_max - feature_min
                    if feature_range > 0:
                        feature_norm = (data[row['feature']] - feature_min) / feature_range
                    else:
                        feature_norm = data[row['feature']]

                    ax.scatter(feature_norm, target_norm, alpha=0.6)

                    z = np.polyfit(feature_norm, target_norm, 1)
                    p = np.poly1d(z)
                    # ax.plot(feature_norm, p(feature_norm), "r--", alpha=0.8)
                else:
                    ax.scatter(data[row['feature']], data[tcol], alpha=0.6)

                    z = np.polyfit(data[row['feature']], data[tcol], 1)
                    p = np.poly1d(z)
                    # ax.plot(data[row['feature']], p(data[row['feature']]), "r--", alpha=0.8)

                ax.set_xlabel(row['feature'], fontsize=10)
                ax.set_ylabel(tcol, fontsize=10)
                p_label = 'p_adj' if use_fdr else 'p'
                ax.set_title(f"ρ={row['spearman_r']:.3f}, {p_label}={row['spearman_p_adj']:.3e}", fontsize=9)
                ax.grid(True, alpha=0.3)

            for idx in range(len(significant), len(axes)):
                axes[idx].axis('off')

            plt.tight_layout()
            plt.show()

        # Return list of (feature, spearman_r) tuples sorted by |spearman_r|.
        significant_tuples = []
        if len(significant) > 0:
            sig = significant.copy()
            sig["spearman_r"] = pd.to_numeric(sig["spearman_r"], errors="coerce")
            sig = sig.dropna(subset=["spearman_r"])
            if len(sig) > 0:
                sig = sig.assign(_abs_r=sig["spearman_r"].abs())
                sig = sig.sort_values("_abs_r", ascending=False).drop(columns=["_abs_r"])
                significant_tuples = list(
                    zip(
                        sig["feature"].astype(str).tolist(),
                        sig["spearman_r"].astype(float).tolist(),
                    )
                )

        corr_df_by_target[tcol] = corr_df
        significant_by_target[tcol] = significant_tuples

    if scalar_target:
        return corr_df_by_target[target_cols[0]], significant_by_target

    return corr_df_by_target, significant_by_target

In [21]:
def stability_selection_features_cv(
    merged_df,  # just one train split dataframe
    target_cols,
    n_subsamples: int = 100,
    sample_fraction: float = 0.5,
    model_type: str = "elasticnet",
    l1_ratio: float = 0.7,
    alpha: float = 0.01,
    coef_threshold: float = 1e-3,
    candidate_features=None,
    random_state: int = 42,
    verbose: bool = True,
):
    """
    Stability selection applied to multiple targets in one train split.

    Returns:
        freq_by_target: dict[target_col, pd.Series]
        selected_by_target: dict[target_col, list[str]]
    """
    from sklearn.linear_model import ElasticNet
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.preprocessing import StandardScaler
    import numpy as np
    import pandas as pd

    if isinstance(target_cols, str):
        target_cols = [target_cols]

    freq_by_target = {}
    selected_by_target = {}
    exclude_cols = target_cols + ["base", "heavy", "light", "name"]

    for target_col in target_cols:

        if candidate_features is not None:
            feat_candidates = [f for f in candidate_features if f in merged_df.columns and f not in exclude_cols]
        else:
            numeric_cols = merged_df.select_dtypes(include=[np.number]).columns
            feat_candidates = [c for c in numeric_cols if c not in exclude_cols]

        if len(feat_candidates) == 0:
            freq_by_target[target_col] = pd.Series(dtype=float)
            selected_by_target[target_col] = []
            continue

        X = merged_df[feat_candidates].apply(pd.to_numeric, errors="coerce").fillna(0).values
        y = pd.to_numeric(merged_df[target_col], errors="coerce").values
        mask = ~np.isnan(y)
        X = X[mask]
        y = y[mask]

        if len(y) < 4:
            freq_by_target[target_col] = pd.Series(dtype=float)
            selected_by_target[target_col] = []
            continue

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        rng = np.random.default_rng(random_state)
        n_feats = len(feat_candidates)
        feature_counts = np.zeros(n_feats)
        subsize = max(2, int(len(y) * sample_fraction))

        for i in range(n_subsamples):
            idx = rng.choice(len(y), size=subsize, replace=False)
            X_sub = X_scaled[idx]
            y_sub = y[idx]

            if model_type.lower() == "elasticnet":
                model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=5000,
                                   random_state=rng.integers(0, 2**31))
                model.fit(X_sub, y_sub)
                non_zero = np.abs(model.coef_) > coef_threshold

            elif model_type.lower() in ("randomforest", "random_forest", "rf"):
                from sklearn.ensemble import RandomForestRegressor
                model = RandomForestRegressor(n_estimators=100, max_features="sqrt",
                                              random_state=rng.integers(0, 2**31))
                model.fit(X_sub, y_sub)
                non_zero = model.feature_importances_ > coef_threshold
            else:
                raise ValueError("model_type must be 'elasticnet' or 'randomforest'")

            feature_counts[non_zero] += 1
            if verbose and (i + 1) % 20 == 0:
                print(f"Stability selection target={target_col}: {i + 1}/{n_subsamples} subsamples")

        freq = pd.Series(feature_counts / n_subsamples, index=feat_candidates)
        freq_by_target[target_col] = freq

        # Elbow selection (Kneedle)
        freq_sorted = freq.sort_values(ascending=False)
        x = np.arange(len(freq_sorted))
        y_vals = freq_sorted.values
        start, end = np.array([x[0], y_vals[0]]), np.array([x[-1], y_vals[-1]])
        line_vec = end - start
        norm = np.linalg.norm(line_vec)
        if norm == 0:
            elbow_idx = 0
        else:
            line_dir = line_vec / norm
            pts = np.stack([x, y_vals], axis=1)
            diffs = pts - start
            proj_lengths = diffs @ line_dir
            proj_points = np.outer(proj_lengths, line_dir)
            orthogonal = diffs - proj_points
            dists = np.linalg.norm(orthogonal, axis=1)
            elbow_idx = int(np.argmax(dists))

        k_feats = max(1, elbow_idx + 1)
        selected_by_target[target_col] = freq_sorted.iloc[:k_feats].index.tolist()

    return freq_by_target, selected_by_target

In [22]:
from sklearn.linear_model import ElasticNet, Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr, pearsonr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def fit_and_compare_models(
    merged_train,
    merged_test,
    target_col,
    feature_list,
    enet_alpha,              # alpha chosen outside (e.g. via CV)
    random_state=42,
    enet_l1_ratio=0.5,
    max_feature_fraction=0.2,
    make_plots=False,
):
    """
    Fit ElasticNet, Ridge, LinearRegression, and RandomForest on TRAIN data,
    evaluate on TEST data.

    - ElasticNet with provided enet_alpha is used on train set to select features:
      max number of features = n_train_samples * max_feature_fraction (top by |coef|).
    - All models are then fit on the selected feature set using train data.
    - Metrics and plots are computed on the test set.

    Returns
    -------
    selected_features : list
        Features kept after ElasticNet-based selection (from train data).
    results_df : pd.DataFrame
        Per-model metrics on the test data:
        ['model', 'r2', 'pearson_r', 'spearman_r', 'spearman_p', 'n_features', 'n_train', 'n_test'].
    """
    # Ensure we don't accidentally include the target itself as a feature (no leakage)
    feature_list = [f for f in feature_list if f not in ("target", target_col, "target_viscosity", "viscosity") and not str(f).startswith("target")]

    X_train = merged_train[feature_list].copy()
    X_test = merged_test[feature_list].copy()

    def _ordinal_encode_train_test(s_train: pd.Series, s_test: pd.Series):
        """Map train-only categories to 0..K-1 (unknown test categories -> NaN)."""
        mask_train = s_train.notna()
        out_train = pd.Series(np.nan, index=s_train.index, dtype=float)
        out_test = pd.Series(np.nan, index=s_test.index, dtype=float)
        if int(mask_train.sum()) == 0:
            return out_train, out_test

        cats = sorted(s_train.loc[mask_train].astype(str).unique().tolist())
        mapping = {cat: i for i, cat in enumerate(cats)}

        out_train.loc[mask_train] = s_train.loc[mask_train].astype(str).map(mapping)
        mask_test = s_test.notna()
        out_test.loc[mask_test] = s_test.loc[mask_test].astype(str).map(mapping)

        return out_train, out_test

    for c in feature_list:
        s_tr = X_train[c]
        s_te = X_test[c]

        # First try numeric conversion (handles numeric columns stored as strings).
        s_tr_num = pd.to_numeric(s_tr, errors="coerce")
        s_te_num = pd.to_numeric(s_te, errors="coerce")

        non_missing_tr = int(s_tr.notna().sum())
        converted_tr = int(s_tr_num.notna().sum())

        if non_missing_tr == 0:
            X_train[c] = s_tr_num
            X_test[c] = s_te_num
            continue

        # If most values can be coerced numerically, treat as numeric.
        if converted_tr >= max(1, int(0.8 * non_missing_tr)):
            X_train[c] = s_tr_num
            X_test[c] = s_te_num
        else:
            X_train[c], X_test[c] = _ordinal_encode_train_test(s_tr, s_te)

    y_train = pd.to_numeric(merged_train[target_col], errors="coerce")
    y_test = pd.to_numeric(merged_test[target_col], errors="coerce")

    train_mask = y_train.notna() & X_train.notna().all(axis=1)
    test_mask = y_test.notna() & X_test.notna().all(axis=1)

    X_train = X_train.loc[train_mask].values
    y_train = y_train.loc[train_mask].values
    X_test = X_test.loc[test_mask].values
    y_test = y_test.loc[test_mask].values

    n_train = len(y_train)
    n_test = len(y_test)

    if n_train < 3 or n_test < 1:
        raise ValueError("Not enough samples to fit/evaluate models.")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # ElasticNet-based feature selection using externally provided alpha (train only)
    enet = ElasticNet(
        alpha=enet_alpha,
        l1_ratio=enet_l1_ratio,
        random_state=random_state,
    )
    enet.fit(X_train_scaled, y_train)

    n_keep = max(1, int(n_train * max_feature_fraction))
    n_keep = min(n_keep, len(feature_list))
    order = np.argsort(np.abs(enet.coef_))[::-1]
    top_indices = order[:n_keep]
    selected_features = [feature_list[i] for i in top_indices]

    X_train_sel = X_train_scaled[:, top_indices]
    X_test_sel = X_test_scaled[:, top_indices]

    # Models: fit on train, evaluate on test
    models = {
        "ElasticNet": ElasticNet(
            alpha=enet_alpha,
            l1_ratio=enet_l1_ratio,
            random_state=random_state,
        ),
        "Ridge": Ridge(alpha=enet_alpha, random_state=random_state),
        "Linear": LinearRegression(),
        "RandomForest": RandomForestRegressor(
            n_estimators=100,
            max_features="sqrt",
            random_state=random_state,
        ),
    }

    results = []
    preds = {}  # for plotting
    for name, model in models.items():
        model.fit(X_train_sel, y_train)
        y_pred = model.predict(X_test_sel)
        preds[name] = y_pred
        r2 = r2_score(y_test, y_pred)
        pr_r, pr_p = pearsonr(y_test, y_pred)
        sp_r, sp_p = spearmanr(y_test, y_pred)
        results.append(
            {
                "model": name,
                "r2": float(r2) if not np.isnan(r2) else np.nan,
                "pearson_r": float(pr_r) if not np.isnan(pr_r) else np.nan,
                "spearman_r": float(sp_r) if not np.isnan(sp_r) else np.nan,
                "spearman_p": float(sp_p) if not np.isnan(sp_p) else np.nan,
                "n_features": len(selected_features),
                "n_train": n_train,
                "n_test": n_test,
            }
        )

    results_df = pd.DataFrame(results)

    # --- Plots (on test set) ---
    # 1) Observed vs predicted (one panel per model)
    if make_plots:
        n_models = len(preds)
        fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 4))
        axes = np.atleast_1d(axes)
        for ax, (name, y_pred) in zip(axes, preds.items()):
            ax.scatter(y_test, y_pred, alpha=0.7, edgecolors="k", linewidths=0.5)
            lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
            ax.plot(lims, lims, "r--", label="y = ŷ")
            ax.set_xlabel("Observed (test)")
            ax.set_ylabel("Predicted (test)")
            ax.set_title(f"{name} (train→test, n_test={n_test})")
            ax.legend(loc="upper left", fontsize=8)
            ax.set_aspect("equal", adjustable="box")
            ax.grid(True, alpha=0.3)
        plt.suptitle(
            f"Test predictions (train fit, no internal CV)\nX_test shape: ({n_test}, {len(selected_features)})",
            fontsize=11,
        )
        plt.tight_layout()
        plt.show()

        # 2) Bar chart: R² and Spearman r per model (on test)
        fig, ax = plt.subplots(figsize=(6, 4))
        x = np.arange(len(results_df))
        w = 0.35
        ax.bar(x - w / 2, results_df["r2"], w, label="R²", color="steelblue", alpha=0.8)
        ax.bar(x + w / 2, results_df["spearman_r"], w, label="Spearman r", color="coral", alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(results_df["model"])
        ax.set_ylabel("Score (test)")
        ax.legend()
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3, axis="y")
        plt.tight_layout()
        plt.show()

    return selected_features, results_df

In [23]:
from sklearn.feature_selection import mutual_info_regression

def reduce_correlated_features(
    split_train_df: "pd.DataFrame",
    target_cols=None,
    feature_set=None,
    correlation_threshold: float = 0.85,
    importance_metric: str = "spearman",
):
    """Reduce highly correlated features within ONE train split.

    This is done per target column, returning a dict:
    ``{target_col: kept_features}``.

    Parameters
    ----------
    split_train_df:
        Training split dataframe.
    target_cols:
        Target column name(s). If None, tries to infer from ``feature_set``
        keys (dict) or from a global ``target_cols`` variable.
    feature_set:
        Optional feature set(s) to restrict the pruning.
        - list[str]: applies to all targets
        - dict[target_col, list[str]]: per-target restriction
        - None: use all numeric columns (excluding target/ID-like columns)

    Returns
    -------
    reduced_features_by_target : dict[str, list[str]]
    """

    import pandas as pd
    import numpy as np
    from scipy.stats import spearmanr
    from scipy.cluster.hierarchy import linkage, fcluster
    from scipy.spatial.distance import squareform

    importance_metric = importance_metric.lower()
    if importance_metric != "spearman":
        raise ValueError("importance_metric must be 'spearman' for hierarchical clustering-based pruning")

    # Normalize target_cols input / inference.
    if target_cols is None:
        if isinstance(feature_set, dict) and len(feature_set) > 0:
            target_cols = list(feature_set.keys())
        elif "target_cols" in globals() and globals().get("target_cols") is not None:
            target_cols = list(globals().get("target_cols"))
        else:
            raise ValueError("target_cols must be provided (or inferred from feature_set keys).")

    if isinstance(target_cols, str):
        target_cols = [target_cols]
    else:
        target_cols = list(target_cols)

    target_cols = [str(t) for t in target_cols]
    if not target_cols:
        raise ValueError("target_cols must be non-empty")

    # Global exclude list to avoid pruning IDs/metadata and avoid selecting targets as features.
    global_exclude_cols = {
        "antibody_id",
        "structure_id",
        "residue_number",
        "n_total_rows",
        "n_filtered_rows",
        "n_beta_sheet_rows",
        "n_exposed_rows",
        "base",
        "heavy",
        "light",
        "name",
        "dataset",
        "antibody_name",
        "index",
        "target",
        "viscosity",
        "target_viscosity",
    }

    # Interpret feature_set as a per-target restriction.
    feature_set_by_target = {}
    if feature_set is None:
        feature_set_by_target = {t: None for t in target_cols}
    elif isinstance(feature_set, dict):
        for t in target_cols:
            feature_set_by_target[t] = feature_set.get(t, None)
    else:
        # Treat list/iterable as one shared feature set.
        shared = list(feature_set)
        feature_set_by_target = {t: shared for t in target_cols}

    reduced_features_by_target = {}

    for target_col in target_cols:
        # Determine candidate features for this target.
        if feature_set_by_target.get(target_col) is None:
            numeric_cols = split_train_df.select_dtypes(include=[np.number]).columns.tolist()
            sig_features = [
                f
                for f in numeric_cols
                if f not in global_exclude_cols
                and f not in target_cols
                and not str(f).startswith("target")
            ]
        else:
            fs = set(feature_set_by_target[target_col])
            sig_features = [
                f
                for f in fs
                if f in split_train_df.columns
                and f not in global_exclude_cols
                and f not in target_cols
                and not str(f).startswith("target")
            ]

        if len(sig_features) < 2:
            reduced_features_by_target[target_col] = sig_features
            continue

        # Prepare numeric data including the target.
        data = split_train_df[sig_features + [target_col]].copy()
        for c in sig_features:
            s = data[c]
            s_num = pd.to_numeric(s, errors="coerce")

            non_missing = int(s.notna().sum())
            converted = int(s_num.notna().sum())

            if non_missing == 0:
                data[c] = s_num
            elif converted >= max(1, int(0.8 * non_missing)):
                data[c] = s_num
            else:
                mask = s.notna()
                cats = sorted(s.loc[mask].astype(str).unique().tolist())
                mapping = {cat: i for i, cat in enumerate(cats)}

                out = pd.Series(np.nan, index=s.index, dtype=float)
                out.loc[mask] = s.loc[mask].astype(str).map(mapping)
                data[c] = out
        data[target_col] = pd.to_numeric(data[target_col], errors="coerce")
        data = data.dropna(how="all")
        if len(data) < 3:
            reduced_features_by_target[target_col] = sig_features
            continue

        # Target importance per feature (to decide which member of a correlated pair to keep).
        target_score = {}
        for f in sig_features:
            sub = data[[f, target_col]].dropna()
            if len(sub) < 3:
                target_score[f] = 0.0
                continue

            r, _ = spearmanr(sub[f], sub[target_col])
            score = float(r) if not np.isnan(r) else 0.0

            target_score[f] = score

        # 1) Spearman feature-feature correlation matrix.
        corr = data[sig_features].corr(method="spearman").abs().fillna(0.0)
        np.fill_diagonal(corr.values, 1.0)

        # 2) Hierarchical clustering on distance = 1 - |corr|.
        dist = 1.0 - corr
        np.fill_diagonal(dist.values, 0.0)

        if len(sig_features) == 2:
            cluster_labels = np.array([1, 1]) if dist.iloc[0, 1] <= (1.0 - correlation_threshold) else np.array([1, 2])
        else:
            condensed = squareform(dist.values, checks=False)
            Z = linkage(condensed, method="average")
            cluster_labels = fcluster(Z, t=(1.0 - correlation_threshold), criterion="distance")

        # 3) One representative per cluster: highest |corr(feature, target)|.
        cluster_to_features = {}
        for feat, label in zip(sig_features, cluster_labels):
            cluster_to_features.setdefault(int(label), []).append(feat)

        kept_features = []
        for label in sorted(cluster_to_features):
            members = cluster_to_features[label]
            best_feat = max(members, key=lambda f: abs(target_score.get(f, 0.0)))
            kept_features.append(best_feat)

        # Keep original feature ordering for downstream reproducibility.
        kept_features = [f for f in sig_features if f in set(kept_features)]
        reduced_features_by_target[target_col] = kept_features

    return reduced_features_by_target

In [24]:
def select_features_forward_sfs(
    split_train_df,
    target_cols,
    candidate_features,
    n_features_to_select=10,
    cv=5,
    scoring="neg_mean_squared_error",
    random_state=42,
    min_improvement=0.01,
    n_jobs=-1,
    model_type="elasticnet",
    rf_n_estimators=200,
    rf_max_depth=None,
    rf_min_samples_leaf=1,
):
    """Manual forward selection per target using custom fold splits."""


    def _build_custom_folds(n_samples, n_splits, seed):
        if n_splits < 2 or n_splits > n_samples:
            return []

        base_size = n_samples // n_splits
        remainder = n_samples % n_splits
        segment_sizes = [base_size + (1 if i < remainder else 0) for i in range(n_splits)]
        boundaries = np.cumsum([0] + segment_sizes)

        rng = np.random.default_rng(seed)
        idx = np.arange(n_samples)
        rng.shuffle(idx)

        folds = []
        for i in range(n_splits):
            val_start, val_end = boundaries[i], boundaries[i + 1]
            val_idx = idx[val_start:val_end]
            train_idx = np.concatenate([idx[:val_start], idx[val_end:]])
            if len(train_idx) == 0 or len(val_idx) == 0:
                continue
            folds.append((train_idx, val_idx))
        return folds

    def _score(y_true, y_pred, metric):
        if metric == "neg_mean_squared_error":
            return -mean_squared_error(y_true, y_pred)
        if metric == "r2":
            return r2_score(y_true, y_pred)
        if metric == "spearman":
            corr, _ = spearmanr(y_true, y_pred)
            return float(corr) if not np.isnan(corr) else -np.inf
        raise ValueError(
            "Unsupported scoring for manual forward selection. "
            "Use 'neg_mean_squared_error', 'r2', or 'spearman'."
        )

    selected_by_target = {}
    model_type_norm = str(model_type).strip().lower()
    if model_type_norm not in {"elasticnet", "randomforest", "random_forest"}:
        raise ValueError("model_type must be 'elasticnet' or 'random_forest'.")

    for target_col in target_cols:
        if isinstance(candidate_features, dict):
            candidate_features_t = list(candidate_features.get(target_col, []))
        else:
            candidate_features_t = list(candidate_features)

        cols = candidate_features_t + [target_col]
        work = split_train_df[cols].replace([np.inf, -np.inf], np.nan).dropna()

        if len(work) < 3 or len(candidate_features_t) == 0:
            selected_by_target[target_col] = []
            continue

        # Ordinal-encode non-numeric candidate features so downstream models can fit.
        for c in candidate_features_t:
            s = work[c]
            s_num = pd.to_numeric(s, errors="coerce")
            non_missing = int(s.notna().sum())
            converted = int(s_num.notna().sum())

            if non_missing == 0:
                work[c] = s_num
            elif converted >= max(1, int(0.8 * non_missing)):
                work[c] = s_num
            else:
                mask = s.notna()
                cats = sorted(s.loc[mask].astype(str).unique().tolist())
                mapping = {cat: i for i, cat in enumerate(cats)}
                out = pd.Series(np.nan, index=s.index, dtype=float)
                out.loc[mask] = s.loc[mask].astype(str).map(mapping)
                work[c] = out

        X = work[candidate_features_t]
        y = work[target_col]
        n_avail = X.shape[1]

        n_select = int(n_features_to_select)
        n_select = max(1, min(n_select, n_avail))

        effective_cv = min(int(cv), len(work))
        if effective_cv < 2:
            selected_by_target[target_col] = list(X.columns[:n_select])
            continue

        folds = _build_custom_folds(len(work), effective_cv, random_state)
        if len(folds) == 0:
            selected_by_target[target_col] = list(X.columns[:n_select])
            continue

        remaining = list(X.columns)
        selected = []
        prev_best_score = -np.inf

        for _ in range(n_select):
            best_feat = None
            best_score = -np.inf

            for feat in remaining:
                candidate_set = selected + [feat]
                fold_scores = []

                for train_idx, val_idx in folds:
                    X_tr = X.iloc[train_idx][candidate_set]
                    y_tr = y.iloc[train_idx]
                    X_val = X.iloc[val_idx][candidate_set]
                    y_val = y.iloc[val_idx]

                    if len(X_tr) < 2 or len(X_val) < 1:
                        continue

                    if model_type_norm == "elasticnet":
                        model = ElasticNet(alpha=0.01, l1_ratio=0.7, max_iter=5000, random_state=random_state)
                    else:
                        model = RandomForestRegressor(
                            n_estimators=rf_n_estimators,
                            max_depth=rf_max_depth,
                            min_samples_leaf=rf_min_samples_leaf,
                            n_jobs=n_jobs,
                            random_state=random_state,
                        )
                    try:
                        model.fit(X_tr, y_tr)
                        y_pred = model.predict(X_val)
                        fold_scores.append(_score(y_val, y_pred, scoring))
                    except Exception:
                        continue

                if len(fold_scores) == 0:
                    continue

                mean_score = float(np.mean(fold_scores))
                if mean_score > best_score:
                    best_score = mean_score
                    best_feat = feat

            if best_feat is None:
                break

            if prev_best_score != -np.inf:
                improvement = best_score - prev_best_score
                if improvement < min_improvement:
                    break

            prev_best_score = best_score

            selected.append(best_feat)
            remaining.remove(best_feat)


            if len(remaining) == 0:
                break

        if len(selected) == 0:
            selected = list(X.columns[:1])

        selected_by_target[target_col] = selected

    return selected_by_target


# Load data

In [25]:
# Experimental datasets + matching descriptor result folders
df_ab21 = load_and_merge(
    exp_csv_path=Path("../data/ab21.csv"),
    results_dir_path=Path("../ab21_results"),
    base="ab21",
)

df_pdgf38 = load_and_merge(
    exp_csv_path=Path("../data/pdgf38.csv"),
    results_dir_path=Path("../pdgf38_results"),
    base="pdgf38",
)

df_garbinski2023_tm1 = load_and_merge(
    exp_csv_path=Path("../data/garbinski2023_tm1.csv"),
    results_dir_path=Path("../garbinski2023_results"),
    base="garbinski2023",
)

# Keep legacy `df` as the garbinski merged dataframe
df = df_garbinski2023_tm1

df_ginkgo = load_and_merge(
    exp_csv_path=Path("../data/ginkgo.csv"),
    results_dir_path=Path("../GINKGO_results"),
    base="GINKGO",
)

df_hutchinson2023enhancement_top200tm1_igg = load_and_merge(
    exp_csv_path=Path("../data/hutchinson2023enhancement_top200tm1_igg.csv"),
    results_dir_path=Path("../hutchinson2023enhancement_results"),
    base="hutchinson2023enhancement",
)

df_jain2017biophysical = load_and_merge(
    exp_csv_path=Path("../data/jain2017biophysical.csv"),
    results_dir_path=Path("../jain2017biophysical_results"),
    base="jain2017biophysical",
)

df_jain2023identifying = load_and_merge(
    exp_csv_path=Path("../data/jain2023identifying.csv"),
    results_dir_path=Path("../jain2023identifying_results"),
    base="jain2023identifying",
)

df_jain2024assessment = load_and_merge(
    exp_csv_path=Path("../data/jain2024assessment.csv"),
    results_dir_path=Path("../jain2024assessment_results"),
    base="jain2024assessment",
)

df_jetha2019homology_RT = load_and_merge(
    exp_csv_path=Path("../data/jetha2019homology_RT.csv"),
    results_dir_path=Path("../jetha2019homology_results"),
    base="jetha2019homology",
)

df_kraft2019herapin_relrt = load_and_merge(
    exp_csv_path=Path("../data/kraft2019herapin_relrt.csv"),
    results_dir_path=Path("../kraft2019herapin_results"),
    base="kraft2019herapin",
)


Merged experimental ab21.csv with results 'ab21': 21 rows
Merged experimental pdgf38.csv with results 'pdgf38': 38 rows
Merged experimental garbinski2023_tm1.csv with results 'garbinski2023': 86 rows
Merged experimental ginkgo.csv with results 'GINKGO': 246 rows
Merged experimental hutchinson2023enhancement_top200tm1_igg.csv with results 'hutchinson2023enhancement': 192 rows
Merged experimental jain2017biophysical.csv with results 'jain2017biophysical': 137 rows
Merged experimental jain2023identifying.csv with results 'jain2023identifying': 115 rows
Merged experimental jain2024assessment.csv with results 'jain2024assessment': 43 rows
Merged experimental jetha2019homology_RT.csv with results 'jetha2019homology': 97 rows
Merged experimental kraft2019herapin_relrt.csv with results 'kraft2019herapin': 128 rows


# Data analytics

In [14]:
feature_cols = df_kraft2019herapin_relrt.columns.values.tolist()
feature_cols.remove("name")
feature_cols.remove("heavy")
feature_cols.remove("light")
feature_cols.remove("target_fitness")
feature_cols.remove("base")
len(feature_cols)

140

In [86]:
features_removed = []
for dataset in [df_kraft2019herapin_relrt, df_jain2024assessment, df_jain2023identifying, df_jain2017biophysical, df_garbinski2023_tm1, df_ab21, df_pdgf38, df_ginkgo, df_hutchinson2023enhancement_top200tm1_igg, df_jetha2019homology_RT]:
    features_removed.extend(remove_low_variance_features(dataset, feature_cols=feature_cols)[1])

In [87]:
features_to_rm = set()
for feature, count in Counter(features_removed).items():
    if count >= 6:
        for dataset in [df_kraft2019herapin_relrt, df_jain2024assessment, df_jain2023identifying, df_jain2017biophysical, df_garbinski2023_tm1, df_ab21, df_pdgf38, df_ginkgo, df_hutchinson2023enhancement_top200tm1_igg, df_jetha2019homology_RT]:
            target_cols = [col for col in dataset.columns.values.tolist() if col.startswith('target_')]
            for target_col in target_cols:
                sp_res = spearmanr(dataset[feature], dataset[target_col])
                if (sp_res.statistic >= 0.2) and (sp_res.pvalue < 0.001):
                    continue
                else:
                    features_to_rm.add(feature)
features_to_rm

{'inter_chain_total_side_rel_sums_aromatic_inter_chain_total_side_rel_sum',
 'inter_chain_total_side_rel_sums_hydrophobic_inter_chain_total_side_rel_sum',
 'inter_chain_total_side_rel_sums_negative_inter_chain_total_side_rel_sum',
 'inter_chain_total_side_rel_sums_polar_inter_chain_total_side_rel_sum',
 'inter_chain_total_side_rel_sums_positive_inter_chain_total_side_rel_sum',
 'sequence_motives_n_motif_ArgLys_CDRs',
 'sequence_motives_n_motif_AsnGly_beta_sheet',
 'sequence_motives_n_motif_AspGlu_inter_chain',
 'sequence_motives_n_motif_AspGly_beta_sheet',
 'sequence_motives_n_motif_AspThr_inter_chain'}

In [51]:
datasets = [df_kraft2019herapin_relrt, df_jain2024assessment, df_jain2023identifying, df_jain2017biophysical, df_garbinski2023_tm1, df_ab21, df_pdgf38, df_ginkgo, df_hutchinson2023enhancement_top200tm1_igg, df_jetha2019homology_RT]
pair_counts = Counter()
threshold= 0.8
top_n = 10
for df_idx, df in enumerate(datasets):
    # Select relevant columns and drop rows with NaNs
    sub_df = df[feature_cols].dropna()
    # Compute Spearman correlation matrix
    corr_matrix = sub_df.corr(method='spearman')
    # Iterate over upper triangle (unique pairs)
    for f1, f2 in itertools.combinations(feature_cols, 2):
        corr_value = corr_matrix.loc[f1, f2]
        if abs(corr_value) >= threshold:
            pair = tuple(sorted((f1, f2)))
            pair_counts[pair] += 1
# Get top N most frequent pairs
top_pairs = pair_counts.most_common(top_n)
print(f"Top {top_n} inter-correlated feature pairs (|Spearman| >= {threshold}):\n")
for (f1, f2), count in top_pairs:
    print(f"{f1} - {f2}: {count}")

Top 10 inter-correlated feature pairs (|Spearman| >= 0.8):

density_metrics_avg_aromatic_all - total_side_rel_sums_aromatic_exposed_total_side_rel_sum: 9
density_metrics_avg_hydrophobic_all - total_side_rel_sums_hydrophobic_exposed_total_side_rel_sum: 9
density_metrics_avg_negative_all - total_side_rel_sums_negative_exposed_total_side_rel_sum: 9
salt_bridges_metrics_avg_salt_all - salt_bridges_metrics_avg_salt_exposed_over_all: 9
charge_metrics_net_charge_by_pH_7.5 - charge_metrics_protein_pi: 9
charge_metrics_net_charge_by_pH_3 - charge_metrics_net_charge_by_pH_7.5: 9
charge_metrics_exposed_net_charge - charge_metrics_exposed_net_charge_from_seq: 9
sequence_motives_n_motif_AspSer - sequence_motives_n_motif_AspSer_exposed: 9
cluster_metrics_positive_exposed_cluster_largest_size - cluster_metrics_positive_exposed_cluster_avg_rel_asa: 8
cluster_metrics_aromatic_exposed_cluster_largest_size - cluster_metrics_aromatic_exposed_cluster_avg_rel_asa: 8


In [29]:
all_sign_feats = []
for dataset in [df_jain2024assessment, df_jain2023identifying, df_jain2017biophysical, df_kraft2019herapin_relrt, df_garbinski2023_tm1, df_ab21, df_pdgf38, df_ginkgo, df_hutchinson2023enhancement_top200tm1_igg, df_jetha2019homology_RT]:
    target_cols = [col for col in dataset.columns.values.tolist() if col.startswith('target_')]
    _, features = calculate_correlations_and_plot(dataset, target_cols, p_threshold=0.05, fdr_alpha=0.05, use_fdr=False, normalize=False, make_plots=False, correlation_threshold=0.0)
    for key in features.keys():
        for el in features[key]:
            all_sign_feats.append(el[0])

[target=target_ACSINS] Total features tested: 135
[target=target_ACSINS] Significant (raw p < 0.05): 11
[target=target_ACSINS] Significant (no FDR): 11
[target=target_ACSINS] After applying |rho| >= 0.0: 11
[target=target_ACSINS] Top correlations by absolute Spearman r (raw p < 0.05):
                                               feature  spearman_r  \
60      salt_bridges_metrics_avg_salt_exposed_over_all    0.413602   
58                   salt_bridges_metrics_avg_salt_all    0.396520   
77   total_side_rel_sums_aromatic_exposed_total_sid...    0.354665   
57         salt_bridges_metrics_number_of_salt_bridges    0.348490   
35                    density_metrics_avg_aromatic_all    0.325728   
63              charge_metrics_dipole_moment_magnitude   -0.328478   
28                       h_bonds_metrics_avg_hbond_cdr   -0.341950   
104               sequence_motives_n_motif_AspAsp_CDRs   -0.361310   
132         sequence_motives_side_asa_sum_motif_AspHis   -0.431903   
97            

In [ ]:
counts = Counter(all_sign_feats)
counts

# Build model

In [30]:
# datasets = [df_ginkgo, df_kraft2019herapin_relrt, df_jain2024assessment, df_jain2023identifying, df_jain2017biophysical, df_garbinski2023_tm1, df_ab21, df_pdgf38]
datasets = [df_ginkgo]
all_stability_features_reduced_across_datasets = []

In [34]:
# Drop rows where ANY target is missing (strict mode)
# Feature-selection pipeline examples:
#   "stability"
#   "correlation"
#   "forward_sfs"
#   "stability->correlation"
#   "correlation->forward_sfs"
feature_selection_pipeline = "correlation"

# Forward SFS params
sfs_inner_cv = 5
sfs_scoring = "spearman"

low_variance_relative_std_threshold = 0.05
low_variance_epsilon = 1e-8

# Global inter-feature de-correlation prefilter (train-only, before pipeline)
intercorr_threshold = 0.80
intercorr_importance_metric = "spearman" # to target

exclude_cols = {
    "base",
    "heavy",
    "light",
    "name",
    "index",
}

# One-hot nominal string/category columns per CV fold (category set = train fold only).
encode_nominal_categoricals = True
# None: auto-detect non-numeric feature columns; or e.g. ["isotype"] to restrict.
categorical_feature_cols = None

pipeline_steps = [s.strip() for s in feature_selection_pipeline.split("->") if s.strip()]
valid_steps = {"stability", "correlation", "forward_sfs"}
if len(pipeline_steps) == 0:
    raise ValueError("feature_selection_pipeline must contain at least one step.")
invalid = [s for s in pipeline_steps if s not in valid_steps]
if invalid:
    raise ValueError(
        f"Invalid pipeline steps: {invalid}. Allowed steps: {sorted(valid_steps)}"
    )

for df in datasets:
    target_cols = [col for col in df.columns.values.tolist() if col.startswith("target_")]
    data = df.dropna(subset=target_cols).copy()
    sfs_n_features_to_select = 0.1 * df.shape[0]

    n_splits = 5
    random_state = 42

    N = len(data)
    if n_splits < 2 or n_splits > N:
        raise ValueError(f"n_splits must be between 2 and N={N}.")

    base_size = N // n_splits
    remainder = N % n_splits
    segment_sizes = [base_size + (1 if i < remainder else 0) for i in range(n_splits)]
    boundaries = np.cumsum([0] + segment_sizes)

    rng = np.random.default_rng(random_state)
    idx = np.arange(N)
    rng.shuffle(idx)

    train_splits = []
    test_splits = []

    for i in range(n_splits):
        test_start, test_end = boundaries[i], boundaries[i + 1]
        test_idx = idx[test_start:test_end]
        train_idx = np.concatenate([idx[:test_start], idx[test_end:]])

        train_df_k = data.iloc[train_idx].copy()
        test_df_k = data.iloc[test_idx].copy()

        train_splits.append([(train_df_k.copy(), t) for t in target_cols])
        test_splits.append([(test_df_k.copy(), t) for t in target_cols])

    features_lowvar = []
    features_intercorr_prefilter_per_split = []
    features_intercorr_prefilter_union_counts = []
    selected_features_per_split = []

    # Optional debug/tracking holders for individual steps
    features_forward_sfs_per_split = []
    features_stability_per_split = []
    features_significant_corrs_reduced_per_split = []

    for k in range(n_splits):
        train_df_k, _ = train_splits[k][0]
        test_df_k, _ = test_splits[k][0]

        candidate_features = [
            c
            for c in train_df_k.columns
            if c not in exclude_cols and c not in target_cols and not str(c).startswith("target")
        ]

        kept_k, removed_k, rel_std_k = remove_low_variance_features(
            X=train_df_k,
            feature_cols=candidate_features,
            relative_std_threshold=low_variance_relative_std_threshold,
            epsilon=low_variance_epsilon,
        )

        if len(kept_k) == 0:
            finite_feats = rel_std_k.index[np.isfinite(rel_std_k.values)].tolist()
            kept_k = finite_feats if len(finite_feats) > 0 else candidate_features[:1]

        kept_k_set = set(kept_k)
        removed_k = [c for c in candidate_features if c not in kept_k_set]

        train_df_k = train_df_k.drop(columns=removed_k, errors="ignore").copy()
        test_df_k = test_df_k.drop(columns=removed_k, errors="ignore").copy()

        # Train-only prefilter before pipeline starts.
        prefilter_by_target_k = reduce_correlated_features(
            split_train_df=train_df_k,
            target_cols=target_cols,
            feature_set=kept_k,
            correlation_threshold=intercorr_threshold,
            importance_metric=intercorr_importance_metric,
        )

        current_features_by_target = {
            t: list(prefilter_by_target_k.get(t, []))
            for t in target_cols
        }

        # Ensure non-empty starting set per target when possible.
        for t in target_cols:
            if len(current_features_by_target[t]) == 0:
                current_features_by_target[t] = kept_k[:1]

        # Sequential feature-selection pipeline.
        for step in pipeline_steps:
            if step == "stability":
                union_feats = sorted({f for t in target_cols for f in current_features_by_target.get(t, [])})
                if len(union_feats) == 0:
                    union_feats = kept_k[:1]

                _, selected_k = stability_selection_features_cv(
                    merged_df=train_df_k,
                    target_cols=target_cols,
                    n_subsamples=100,
                    sample_fraction=0.5,
                    model_type="elasticnet",
                    l1_ratio=0.7,
                    alpha=0.01,
                    coef_threshold=0.05,
                    candidate_features=union_feats,
                    random_state=42,
                    verbose=True,
                )

                next_features = {}
                for t in target_cols:
                    vals = list(selected_k.get(t, []))
                    next_features[t] = vals if len(vals) > 0 else list(current_features_by_target.get(t, []))
                current_features_by_target = next_features
                features_stability_per_split.append(current_features_by_target)

            elif step == "correlation":
                corr_df_k, significant_k = calculate_correlations_and_plot(
                    merged_df=train_df_k,
                    target_col=target_cols,
                    p_threshold=0.05,
                    fdr_alpha=0.05,
                    use_fdr=True,
                    normalize=False,
                    make_plots=False,
                    correlation_threshold=0,
                )

                sig_features_for_split = {}
                for t in target_cols:
                    sig_t = [el[0] for el in significant_k.get(t, [])]
                    allowed_t = set(current_features_by_target.get(t, []))
                    sig_features_for_split[t] = [f for f in sig_t if f in allowed_t]

                reduced_corr = reduce_correlated_features(
                    split_train_df=train_df_k,
                    target_cols=target_cols,
                    feature_set=sig_features_for_split,
                    correlation_threshold=0.8,
                    importance_metric="spearman",
                )

                next_features = {}
                for t in target_cols:
                    vals = list(reduced_corr.get(t, []))
                    next_features[t] = vals if len(vals) > 0 else list(current_features_by_target.get(t, []))
                current_features_by_target = next_features
                features_significant_corrs_reduced_per_split.append(current_features_by_target)

            else:  # step == "forward_sfs"
                selected_sfs_k = select_features_forward_sfs(
                    split_train_df=train_df_k,
                    target_cols=target_cols,
                    candidate_features=current_features_by_target,
                    n_features_to_select=sfs_n_features_to_select,
                    model_type="elasticnet",
                    cv=sfs_inner_cv,
                    scoring=sfs_scoring,
                    random_state=random_state,
                    min_improvement=0.01,
                    n_jobs=-1,
                )

                next_features = {}
                for t in target_cols:
                    vals = list(selected_sfs_k.get(t, []))
                    next_features[t] = vals if len(vals) > 0 else list(current_features_by_target.get(t, []))
                current_features_by_target = next_features
                features_forward_sfs_per_split.append(current_features_by_target)

        train_splits[k] = [(train_df_k.copy(), t) for t in target_cols]
        test_splits[k] = [(test_df_k.copy(), t) for t in target_cols]

        features_lowvar.append(kept_k)
        features_intercorr_prefilter_per_split.append(prefilter_by_target_k)
        features_intercorr_prefilter_union_counts.append(
            len(sorted({f for t in target_cols for f in prefilter_by_target_k.get(t, [])}))
        )

        selected_features_per_split.append(current_features_by_target)

    print("Feature-selection pipeline:", " -> ".join(pipeline_steps))
    print("Low-variance kept feature counts per split:", [len(x) for x in features_lowvar])
    print("Inter-corr prefilter kept feature counts per split:", features_intercorr_prefilter_union_counts)
    print("Forward SFS kept feature counts per split:", [len(x[t]) for x in features_forward_sfs_per_split])
    print(
        f"Created {n_splits} splits: "
        f"test sizes {segment_sizes}, train sizes {[N - s for s in segment_sizes]}."
    )


[target=target_SEC_Monomer] Total features tested: 125
[target=target_SEC_Monomer] Significant (raw p < 0.05): 6
[target=target_SEC_Monomer] Significant after FDR correction (adj p < 0.05): 0
[target=target_SEC_Monomer] After applying |rho| >= 0: 0
[target=target_SEC_Monomer] Top correlations by absolute Spearman r (FDR-significant only):
(no significant correlations)
[target=target_SMAC] Total features tested: 125
[target=target_SMAC] Significant (raw p < 0.05): 19
[target=target_SMAC] Significant after FDR correction (adj p < 0.05): 6
[target=target_SMAC] After applying |rho| >= 0: 6
[target=target_SMAC] Top correlations by absolute Spearman r (FDR-significant only):
                                              feature  spearman_r  spearman_p  \
45                            cdr_metrics_cdr3_length    0.540993    0.000017   
55             wcn_metrics_avg_wcn_interface_over_all    0.446234    0.000567   
80  total_side_rel_sums_aromatic_exposed_total_sid...    0.430218    0.000935  

In [33]:
for target_col in target_cols:
    results_per_split = []

    for split_idx in range(n_splits):
        train_df, _ = train_splits[split_idx][0]
        test_df, _ = test_splits[split_idx][0]

        # selected_features_per_split stores dict[target_col] -> list[str]
        feature_map = selected_features_per_split[split_idx]
        features = feature_map.get(target_col, [])

        if len(features) == 0:
            raise ValueError(
                f"No selected features for target={target_col}, split={split_idx}. "
                "Check feature-selection outputs before model fitting."
            )

        enet_alpha = 0.01

        selected_features, results_df = fit_and_compare_models(
            merged_train=train_df,
            merged_test=test_df,
            target_col=target_col,
            feature_list=features,
            enet_alpha=enet_alpha,
            enet_l1_ratio=0.7,
            max_feature_fraction=0.1,
            make_plots=False,
        )
        print(selected_features)

        results_per_split.append(results_df)

    results_agg = pd.concat(results_per_split, ignore_index=True)
    summary = results_agg.groupby("model").agg({"pearson_r": "mean", "spearman_r": "mean", "r2": "mean"}).reset_index()
    print("Mean test metrics across splits:")
    print(target_col)
    print(summary)

['cluster_metrics_ripley_k_positive', 'density_metrics_avg_hydrophobic_cdr_over_cdr', 'cluster_metrics_aromatic_exposed_cluster_avg_rel_asa']
['other_sasa_metrics_inter_chain_buried_sasa', 'cluster_metrics_ripley_k_positive', 'charge_metrics_unweighted_scm_score_by_pH_3', 'h_bonds_metrics_avg_hbond_inter_chain', 'cluster_metrics_pos_ann_index']
['charge_metrics_net_charge_cdr_from_pka', 'cluster_metrics_negative_exposed_cluster_largest_size']
['cluster_metrics_neg_ann_index', 'cdr_metrics_ratio_hydrophobic_to_polar_CDRs', 'cluster_metrics_aromatic_exposed_cluster_avg_rel_asa', 'cdr_metrics_cdr3_length', 'sequence_motives_n_motif_ArgLys_CDRs']
['h_bonds_metrics_avg_hbond_energy_dssp_weighted_buried_over_all', 'cluster_metrics_ripley_k_positive', 'cluster_metrics_aromatic_exposed_cluster_avg_rel_asa', 'charge_metrics_sap_neg_charge_score', 'sequence_motives_n_motif_AspAsp_exposed']
Mean test metrics across splits:
target_SEC_Monomer
          model  pearson_r  spearman_r        r2
0    E

In [ ]:
['cluster_metrics_neg_ann_index', 'salt_bridges_metrics_number_of_salt_bridges', 'charge_metrics_light_charge_pH74']
['cluster_metrics_positive_exposed_cluster_largest_size', 'salt_bridges_metrics_number_of_salt_bridges']
['cluster_metrics_positive_exposed_cluster_largest_size', 'h_bonds_metrics_avg_hbond_cdr', 'charge_metrics_dipole_moment_magnitude']
['charge_metrics_heavy_charge_pH74', 'salt_bridges_metrics_avg_salt_inter_chain', 'density_metrics_avg_negative_cdr_over_cdr']
['cluster_metrics_pnc_all_surface_exposed', 'salt_bridges_metrics_number_of_salt_bridges', 'h_bonds_metrics_avg_hbond_cdr']
Mean test metrics across splits:
target_viscosity
          model  pearson_r  spearman_r
0    ElasticNet   0.831759    0.674496
1        Linear   0.831529    0.664972
2  RandomForest   0.845018    0.622115
3         Ridge   0.831555    0.664972

# Propermab

In [205]:
df = pd.read_csv("../GINKGO_propermab/features.csv")
exp_df = pd.read_csv("../data/ginkgo.csv")
exp_df['name'] = exp_df['name'].astype(str)
df['name'] = df['pdb_file'].str.split("/").str[-1].str.split('.').apply(lambda x: x[0])
df = df.drop(columns=['pdb_file', 'error'], errors='ignore')
df = df.merge(exp_df, on='name', how='inner')
target_cols = [col for col in df.columns.values.tolist() if col.startswith('target_')]
datasets = [df]

In [221]:
merged = df.merge(df_jain2024assessment, on='name', how='inner').dropna()

In [223]:
spearmanr(merged['scm'], merged['charge_metrics_scm_propermab_score_by_pH_7.5'])

SignificanceResult(statistic=0.8256097560975612, pvalue=3.084368000212194e-11)

In [ ]:
# Drop rows where ANY target is missing (strict mode)
# Feature-selection pipeline examples:
#   "stability"
#   "correlation"
#   "forward_sfs"
#   "stability->correlation"
#   "correlation->forward_sfs"
feature_selection_pipeline = "stability"

# Forward SFS params
sfs_inner_cv = 5
sfs_scoring = "spearman"

low_variance_relative_std_threshold = 0.05
low_variance_epsilon = 1e-8

# Global inter-feature de-correlation prefilter (train-only, before pipeline)
intercorr_threshold = 0.85
intercorr_importance_metric = "spearman"

exclude_cols = {
    "base",
    "heavy",
    "light",
    "name",
    "index",
}

# One-hot nominal string/category columns per CV fold (category set = train fold only).
encode_nominal_categoricals = True
# None: auto-detect non-numeric feature columns; or e.g. ["isotype"] to restrict.
categorical_feature_cols = None

pipeline_steps = [s.strip() for s in feature_selection_pipeline.split("->") if s.strip()]
valid_steps = {"stability", "correlation", "forward_sfs"}
if len(pipeline_steps) == 0:
    raise ValueError("feature_selection_pipeline must contain at least one step.")
invalid = [s for s in pipeline_steps if s not in valid_steps]
if invalid:
    raise ValueError(
        f"Invalid pipeline steps: {invalid}. Allowed steps: {sorted(valid_steps)}"
    )

for df in datasets:
    target_cols = [col for col in df.columns.values.tolist() if col.startswith("target_")]
    data = df.dropna(subset=target_cols).copy()
    sfs_n_features_to_select = 0.1 * df.shape[0]

    n_splits = 5
    random_state = 42

    N = len(data)
    if n_splits < 2 or n_splits > N:
        raise ValueError(f"n_splits must be between 2 and N={N}.")

    base_size = N // n_splits
    remainder = N % n_splits
    segment_sizes = [base_size + (1 if i < remainder else 0) for i in range(n_splits)]
    boundaries = np.cumsum([0] + segment_sizes)

    rng = np.random.default_rng(random_state)
    idx = np.arange(N)
    rng.shuffle(idx)

    train_splits = []
    test_splits = []

    for i in range(n_splits):
        test_start, test_end = boundaries[i], boundaries[i + 1]
        test_idx = idx[test_start:test_end]
        train_idx = np.concatenate([idx[:test_start], idx[test_end:]])

        train_df_k = data.iloc[train_idx].copy()
        test_df_k = data.iloc[test_idx].copy()

        train_splits.append([(train_df_k.copy(), t) for t in target_cols])
        test_splits.append([(test_df_k.copy(), t) for t in target_cols])

    features_lowvar = []
    features_intercorr_prefilter_per_split = []
    features_intercorr_prefilter_union_counts = []
    selected_features_per_split = []

    # Optional debug/tracking holders for individual steps
    features_forward_sfs_per_split = []
    features_stability_per_split = []
    features_significant_corrs_reduced_per_split = []

    for k in range(n_splits):
        train_df_k, _ = train_splits[k][0]
        test_df_k, _ = test_splits[k][0]

        candidate_features = [
            c
            for c in train_df_k.columns
            if c not in exclude_cols and c not in target_cols and not str(c).startswith("target")
        ]

        kept_k, removed_k, rel_std_k = remove_low_variance_features(
            X=train_df_k,
            feature_cols=candidate_features,
            relative_std_threshold=low_variance_relative_std_threshold,
            epsilon=low_variance_epsilon,
        )

        if len(kept_k) == 0:
            finite_feats = rel_std_k.index[np.isfinite(rel_std_k.values)].tolist()
            kept_k = finite_feats if len(finite_feats) > 0 else candidate_features[:1]

        kept_k_set = set(kept_k)
        removed_k = [c for c in candidate_features if c not in kept_k_set]

        train_df_k = train_df_k.drop(columns=removed_k, errors="ignore").copy()
        test_df_k = test_df_k.drop(columns=removed_k, errors="ignore").copy()

        # Train-only prefilter before pipeline starts.
        prefilter_by_target_k = reduce_correlated_features(
            split_train_df=train_df_k,
            target_cols=target_cols,
            feature_set=kept_k,
            correlation_threshold=intercorr_threshold,
            importance_metric=intercorr_importance_metric,
        )

        current_features_by_target = {
            t: list(prefilter_by_target_k.get(t, []))
            for t in target_cols
        }

        # Ensure non-empty starting set per target when possible.
        for t in target_cols:
            if len(current_features_by_target[t]) == 0:
                current_features_by_target[t] = kept_k[:1]

        # Sequential feature-selection pipeline.
        for step in pipeline_steps:
            if step == "stability":
                union_feats = sorted({f for t in target_cols for f in current_features_by_target.get(t, [])})
                if len(union_feats) == 0:
                    union_feats = kept_k[:1]

                _, selected_k = stability_selection_features_cv(
                    merged_df=train_df_k,
                    target_cols=target_cols,
                    n_subsamples=100,
                    sample_fraction=0.5,
                    model_type="elasticnet",
                    l1_ratio=0.7,
                    alpha=0.01,
                    coef_threshold=0.05,
                    candidate_features=union_feats,
                    random_state=42,
                    verbose=True,
                )

                next_features = {}
                for t in target_cols:
                    vals = list(selected_k.get(t, []))
                    next_features[t] = vals if len(vals) > 0 else list(current_features_by_target.get(t, []))
                current_features_by_target = next_features
                features_stability_per_split.append(current_features_by_target)

            elif step == "correlation":
                corr_df_k, significant_k = calculate_correlations_and_plot(
                    merged_df=train_df_k,
                    target_col=target_cols,
                    p_threshold=0.05,
                    fdr_alpha=0.05,
                    use_fdr=True,
                    normalize=False,
                    make_plots=False,
                    correlation_threshold=0,
                )

                sig_features_for_split = {}
                for t in target_cols:
                    sig_t = [el[0] for el in significant_k.get(t, [])]
                    allowed_t = set(current_features_by_target.get(t, []))
                    sig_features_for_split[t] = [f for f in sig_t if f in allowed_t]

                reduced_corr = reduce_correlated_features(
                    split_train_df=train_df_k,
                    target_cols=target_cols,
                    feature_set=sig_features_for_split,
                    correlation_threshold=0.8,
                    importance_metric="spearman",
                )

                next_features = {}
                for t in target_cols:
                    vals = list(reduced_corr.get(t, []))
                    next_features[t] = vals if len(vals) > 0 else list(current_features_by_target.get(t, []))
                current_features_by_target = next_features
                features_significant_corrs_reduced_per_split.append(current_features_by_target)

            else:  # step == "forward_sfs"
                selected_sfs_k = select_features_forward_sfs(
                    split_train_df=train_df_k,
                    target_cols=target_cols,
                    candidate_features=current_features_by_target,
                    n_features_to_select=sfs_n_features_to_select,
                    model_type="elasticnet",
                    cv=sfs_inner_cv,
                    scoring=sfs_scoring,
                    random_state=random_state,
                    min_improvement=0.02,
                    n_jobs=-1,
                )

                next_features = {}
                for t in target_cols:
                    vals = list(selected_sfs_k.get(t, []))
                    next_features[t] = vals if len(vals) > 0 else list(current_features_by_target.get(t, []))
                current_features_by_target = next_features
                features_forward_sfs_per_split.append(current_features_by_target)

        train_splits[k] = [(train_df_k.copy(), t) for t in target_cols]
        test_splits[k] = [(test_df_k.copy(), t) for t in target_cols]

        features_lowvar.append(kept_k)
        features_intercorr_prefilter_per_split.append(prefilter_by_target_k)
        features_intercorr_prefilter_union_counts.append(
            len(sorted({f for t in target_cols for f in prefilter_by_target_k.get(t, [])}))
        )

        selected_features_per_split.append(current_features_by_target)

    print("Feature-selection pipeline:", " -> ".join(pipeline_steps))
    print("Low-variance kept feature counts per split:", [len(x) for x in features_lowvar])
    print("Inter-corr prefilter kept feature counts per split:", features_intercorr_prefilter_union_counts)
    print("Forward SFS kept feature counts per split:", [len(x[t]) for x in features_forward_sfs_per_split])
    print(
        f"Created {n_splits} splits: "
        f"test sizes {segment_sizes}, train sizes {[N - s for s in segment_sizes]}."
    )


Stability selection target=target_SEC_Monomer: 20/100 subsamples
Stability selection target=target_SEC_Monomer: 40/100 subsamples
Stability selection target=target_SEC_Monomer: 60/100 subsamples
Stability selection target=target_SEC_Monomer: 80/100 subsamples
Stability selection target=target_SEC_Monomer: 100/100 subsamples
Stability selection target=target_SMAC: 20/100 subsamples
Stability selection target=target_SMAC: 40/100 subsamples
Stability selection target=target_SMAC: 60/100 subsamples
Stability selection target=target_SMAC: 80/100 subsamples
Stability selection target=target_SMAC: 100/100 subsamples
Stability selection target=target_HIC: 20/100 subsamples
Stability selection target=target_HIC: 40/100 subsamples
Stability selection target=target_HIC: 60/100 subsamples
Stability selection target=target_HIC: 80/100 subsamples
Stability selection target=target_HIC: 100/100 subsamples
Stability selection target=target_HAC: 20/100 subsamples
Stability selection target=target_HAC: 4

In [209]:
for target_col in target_cols:
    results_per_split = []

    for split_idx in range(n_splits):
        train_df, _ = train_splits[split_idx][0]
        test_df, _ = test_splits[split_idx][0]

        # selected_features_per_split stores dict[target_col] -> list[str]
        feature_map = selected_features_per_split[split_idx]
        features = feature_map.get(target_col, [])

        if len(features) == 0:
            raise ValueError(
                f"No selected features for target={target_col}, split={split_idx}. "
                "Check feature-selection outputs before model fitting."
            )

        enet_alpha = 0.01

        selected_features, results_df = fit_and_compare_models(
            merged_train=train_df,
            merged_test=test_df,
            target_col=target_col,
            feature_list=features,
            enet_alpha=enet_alpha,
            enet_l1_ratio=0.7,
            max_feature_fraction=0.15,
            make_plots=False,
        )
        print(selected_features)

        results_per_split.append(results_df)

    results_agg = pd.concat(results_per_split, ignore_index=True)
    summary = results_agg.groupby("model").agg({"pearson_r": "mean", "spearman_r": "mean", "r2": "mean"}).reset_index()
    print("Mean test metrics across splits:")
    print(target_col)
    print(summary)

['dipole_moment', 'neg_ann_index', 'net_charge_cdr', 'hyd_patch_area_cdr', 'aromatic_cdr', 'hyd_patch_area', 'exposed_net_charge_cdr', 'net_charge']
['net_charge_cdr', 'hyd_patch_area_cdr', 'dipole_moment', 'hyd_patch_area', 'net_charge', 'neg_ann_index', 'aromatic_cdr', 'aromatic_asa']
['net_charge', 'hyd_patch_area_cdr', 'dipole_moment', 'hyd_patch_area', 'pos_patch_area_cdr', 'neg_ann_index', 'aromatic_cdr', 'exposed_aromatic']
['hyd_patch_area', 'neg_ann_index', 'hyd_patch_area_cdr', 'net_charge_cdr', 'aromatic_cdr', 'dipole_moment', 'cdr_h3_length', 'scm']
['exposed_aromatic', 'Fv_chml', 'hyd_patch_area']
Mean test metrics across splits:
target_SEC_Monomer
          model  pearson_r  spearman_r        r2
0    ElasticNet   0.157618    0.154286 -2.713422
1        Linear   0.159445    0.158681 -2.738496
2  RandomForest   0.108743    0.111209 -4.841490
3         Ridge   0.159367    0.158681 -2.737893
['hyd_patch_area_cdr', 'net_charge_cdr', 'hyd_patch_area', 'exposed_net_charge_cdr', 

In [38]:
# Diagnostic: verify categorical columns get kept/used (ordinal-encoded) per split
# This re-runs the same train-fold-only preprocessing used in `fit_and_compare_models`
# and reports which columns become numeric vs ordinal-coded, plus how many rows are dropped.

import numpy as np
import pandas as pd

# ---- user params ----
# Pick one split and target to inspect
split_idx = 0
# If you already have `target_cols` in scope, set target_col = target_cols[0] (or pick explicitly)
target_col = target_cols[0] if 'target_cols' in globals() else None

# Hyperparams should match the call you made in the training loop
enet_alpha = 0.01
enet_l1_ratio = 0.7
max_feature_fraction = 0.15
random_state = 42

if target_col is None:
    raise ValueError("Set `target_col` (and ensure `target_cols` exists in this notebook).")

if 'train_splits' not in globals() or 'test_splits' not in globals():
    raise RuntimeError("This diagnostic cell requires `train_splits` and `test_splits` variables from the notebook.")

# Expect train_splits[k] = [(train_df_k, target_col_k), ...] (per target)
# and selected_features_per_split[k] = {target_col: [feature names]}
train_df = None
test_df = None

# Pull the exact (df, tcol) tuple for this target when possible.
try:
    train_entry = next((x for x in train_splits[split_idx] if x[1] == target_col), None)
    test_entry = next((x for x in test_splits[split_idx] if x[1] == target_col), None)
    if train_entry is not None and test_entry is not None:
        train_df = train_entry[0]
        test_df = test_entry[0]
except Exception:
    # Fallback: take the first tuple (should be same underlying split df)
    train_df = train_splits[split_idx][0][0]
    test_df = test_splits[split_idx][0][0]

# Use the pre-pruning feature list if available; otherwise ask for it
if 'selected_features_per_split' in globals():
    feature_map = selected_features_per_split[split_idx]
    feature_list = feature_map.get(target_col, [])
else:
    feature_list = []

if not feature_list:
    raise RuntimeError(
        "No `feature_list` found for the chosen (split_idx, target_col). "
        "Make sure `selected_features_per_split` exists and contains that target."
    )

# Match fit_and_compare_models feature filtering
feature_list = [
    f for f in feature_list
    if f not in ("target", target_col, "target_viscosity", "viscosity")
    and not str(f).startswith("target")
]

X_train = train_df[feature_list].copy()
X_test = test_df[feature_list].copy()

# Same encoding heuristic as in `fit_and_compare_models`

def _ordinal_encode_train_test(s_train: pd.Series, s_test: pd.Series):
    """Map train-only categories to 0..K-1 (unknown test categories -> NaN)."""
    mask_train = s_train.notna()
    out_train = pd.Series(np.nan, index=s_train.index, dtype=float)
    out_test = pd.Series(np.nan, index=s_test.index, dtype=float)
    if int(mask_train.sum()) == 0:
        return out_train, out_test, {}
    cats = sorted(s_train.loc[mask_train].astype(str).unique().tolist())
    mapping = {cat: i for i, cat in enumerate(cats)}
    out_train.loc[mask_train] = s_train.loc[mask_train].astype(str).map(mapping)
    mask_test = s_test.notna()
    out_test.loc[mask_test] = s_test.loc[mask_test].astype(str).map(mapping)
    return out_train, out_test, mapping

# Track decisions
encisions = []
category_maps = {}

for c in feature_list:
    s_tr = X_train[c]
    s_te = X_test[c]

    s_tr_num = pd.to_numeric(s_tr, errors="coerce")
    s_te_num = pd.to_numeric(s_te, errors="coerce")

    non_missing_tr = int(s_tr.notna().sum())
    converted_tr = int(s_tr_num.notna().sum())

    if non_missing_tr == 0:
        # Column is all missing in train -> keep numeric coercion result (all NaN)
        X_train[c] = s_tr_num
        X_test[c] = s_te_num
        encisions.append({
            'feature': c,
            'encoding': 'numeric(all_missing_train)',
            'non_missing_train': non_missing_tr,
            'converted_train': converted_tr,
            'n_categories_train': 0,
        })
        continue

    threshold = max(1, int(0.8 * non_missing_tr))

    if converted_tr >= threshold:
        X_train[c] = s_tr_num
        X_test[c] = s_te_num
        encisions.append({
            'feature': c,
            'encoding': 'numeric',
            'non_missing_train': non_missing_tr,
            'converted_train': converted_tr,
            'n_categories_train': 0,
        })
    else:
        X_train[c], X_test[c], mapping = _ordinal_encode_train_test(s_tr, s_te)
        category_maps[c] = mapping
        encisions.append({
            'feature': c,
            'encoding': f'ordinal(K={len(mapping)})',
            'non_missing_train': non_missing_tr,
            'converted_train': converted_tr,
            'n_categories_train': len(mapping),
        })

# Encode targets

y_train = pd.to_numeric(train_df[target_col], errors="coerce")
y_test = pd.to_numeric(test_df[target_col], errors="coerce")

# Row masks exactly like fit_and_compare_models
train_mask = y_train.notna() & X_train.notna().all(axis=1)
test_mask = y_test.notna() & X_test.notna().all(axis=1)

n_train = int(train_mask.sum())
n_test = int(test_mask.sum())

print(f"Diagnostics for split_idx={split_idx}, target_col={target_col}")
print(f"n_features(input)={len(feature_list)}")
print(f"train rows after masks: n_train={n_train} (kept out of {len(train_df)})")
print(f"test rows after masks:  n_test={n_test} (kept out of {len(test_df)})")
print()

# Show encoding decisions (sorted with ordinal first)
enc_df = pd.DataFrame(encisions)
if not enc_df.empty:
    enc_df['encoding_kind'] = enc_df['encoding'].str.contains('ordinal').map({True:'ordinal', False:'numeric'})
    enc_df = enc_df.sort_values(['encoding_kind','n_categories_train','feature'], ascending=[False,False,True])
    print(enc_df.to_string(index=False))
print()

# For ordinal columns, report how many test values become unknown->NaN
ordinal_cols = [d['feature'] for d in encisions if str(d['encoding']).startswith('ordinal')]
unknown_reports = []
for c in ordinal_cols:
    mapping = category_maps.get(c, {})
    s_tr = train_df[c]
    s_te = test_df[c]
    mask_te = s_te.notna()
    unknown_mask = mask_te & s_te.astype(str).map(mapping).isna()
    unknown_reports.append({
        'feature': c,
        'K_train': len(mapping),
        'unknown_test_values': int(unknown_mask.sum()),
        'test_non_missing': int(mask_te.sum()),
        'test_encoded_nan_total': int(pd.to_numeric(X_test[c], errors='coerce').isna().sum()),
    })

if unknown_reports:
    print('Unknown/NaN introduced by ordinal encoding (test fold):')
    print(pd.DataFrame(unknown_reports).to_string(index=False))
    print()

# Re-run the ElasticNet selection step to see which features are actually used
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet

X_train_sel_source = X_train.loc[train_mask].values
X_test_sel_source = X_test.loc[test_mask].values
y_train_sel = y_train.loc[train_mask].values
y_test_sel = y_test.loc[test_mask].values

if n_train < 3 or n_test < 1:
    print("Skipping ElasticNet selection because n_train/n_test too small.")
else:
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_sel_source)
    X_test_scaled = scaler.transform(X_test_sel_source)

    enet = ElasticNet(alpha=enet_alpha, l1_ratio=enet_l1_ratio, random_state=random_state)
    enet.fit(X_train_scaled, y_train_sel)

    n_keep = max(1, int(n_train * max_feature_fraction))
    n_keep = min(n_keep, len(feature_list))

    order = np.argsort(np.abs(enet.coef_))[::-1]
    top_indices = order[:n_keep]
    selected_features = [feature_list[i] for i in top_indices]

    # Print selected and encoding types
    enc_map = {r['feature']: r['encoding'] for _, r in enc_df.iterrows()} if not enc_df.empty else {}
    selected_rows = []
    for f in selected_features:
        selected_rows.append({
            'selected_feature': f,
            'encoding': enc_map.get(f, ''),
        })

    print(f"ElasticNet selected n_keep={n_keep} features (used for modeling):")
    print(pd.DataFrame(selected_rows).to_string(index=False))


Diagnostics for split_idx=0, target_col=target_SEC_Monomer
n_features(input)=93
train rows after masks: n_train=56 (kept out of 56)
test rows after masks:  n_test=14 (kept out of 14)

                                                          feature encoding  non_missing_train  converted_train  n_categories_train encoding_kind
                   beta_sheet_metrics_fraction_gln_asn_beta_sheet  numeric                 56               56                   0       numeric
     beta_sheet_metrics_hydrophobic_beta_sheet_kyte_doolittle_sum  numeric                 56               56                   0       numeric
                          buried_metrics_fraction_negative_buried  numeric                 56               56                   0       numeric
                          buried_metrics_fraction_positive_buried  numeric                 56               56                   0       numeric
                                          cdr_metrics_cdr3_length  numeric                 

In [39]:
# Diagnostic-only: track feature elimination and print removed features
# Focus: why categorical string columns (e.g. feature_hc_subtype / feature_lc_subtype)
# disappear before modeling.

import numpy as np
import pandas as pd

# ---- user params ----
split_idx = 0
# Ensure `target_cols` exists or set explicitly:
# target_col = "target_HAC"  # example
if 'target_cols' in globals() and globals().get('target_cols') is not None:
    target_col = globals()['target_cols'][0]
else:
    raise RuntimeError("Set `target_col` manually or run earlier cells that define `target_cols`. ")

# Pipeline / thresholds (use globals when available)
low_variance_relative_std_threshold = globals().get('low_variance_relative_std_threshold', 0.05)
low_variance_epsilon = globals().get('low_variance_epsilon', 1e-8)
intercorr_threshold = globals().get('intercorr_threshold', 0.80)
intercorr_importance_metric = globals().get('intercorr_importance_metric', 'spearman')

pipeline_steps_diag = globals().get('pipeline_steps', None)
if pipeline_steps_diag is None:
    feature_selection_pipeline = globals().get('feature_selection_pipeline', 'correlation')
    pipeline_steps_diag = [s.strip() for s in str(feature_selection_pipeline).split('->') if s.strip()]

exclude_cols_diag = set(globals().get('exclude_cols', {'base','heavy','light','name','index'}))

if 'train_splits' not in globals() or 'test_splits' not in globals():
    raise RuntimeError("This diagnostic cell needs `train_splits` and `test_splits` defined.")

# Pull the exact (df, tcol) tuple for this target when possible.
try:
    train_entry = next((x for x in train_splits[split_idx] if x[1] == target_col), None)
    test_entry = next((x for x in test_splits[split_idx] if x[1] == target_col), None)
    if train_entry is None or test_entry is None:
        train_df = train_splits[split_idx][0][0]
        test_df = test_splits[split_idx][0][0]
    else:
        train_df = train_entry[0]
        test_df = test_entry[0]
except Exception:
    train_df = train_splits[split_idx][0][0]
    test_df = test_splits[split_idx][0][0]

# Determine targets present in this split
target_cols_diag = [c for c in train_df.columns.values.tolist() if str(c).startswith('target_')]

# Starting candidate features (same logic as main feature-selection cell)
candidate_features_diag = [
    c
    for c in train_df.columns
    if c not in exclude_cols_diag
    and c not in target_cols_diag
    and not str(c).startswith('target')
]

print("\n==== Feature elimination tracking (diagnostic-only) ====")
print(f"split_idx={split_idx}, target_col={target_col}")
print(f"candidate_features_diag count={len(candidate_features_diag)}")
for special in ['feature_hc_subtype','feature_lc_subtype']:
    if special in train_df.columns:
        print(f"Special present: {special} dtype={train_df[special].dtype}")
    else:
        print(f"Special missing in this split: {special}")


def _print_eliminated(stage_name: str, before_feats, after_feats, max_show: int = 80):
    eliminated = sorted(set(before_feats) - set(after_feats))
    print(f"[diag] {stage_name}: eliminated {len(eliminated)} features for target={target_col}")
    for special in ['feature_hc_subtype','feature_lc_subtype']:
        if special in eliminated:
            print(f"[diag]  - eliminated special: {special}")
    if len(eliminated) <= max_show:
        print(eliminated)
    else:
        print(f"showing first {max_show}: {eliminated[:max_show]} ...")


# ---- Stage 1: low-variance filter ----
kept_k, removed_k, _rel_std = remove_low_variance_features(
    X=train_df,
    feature_cols=candidate_features_diag,
    relative_std_threshold=low_variance_relative_std_threshold,
    epsilon=low_variance_epsilon,
)
_print_eliminated('low_variance_filter', candidate_features_diag, kept_k)

train_df_k = train_df.drop(columns=removed_k, errors='ignore').copy()

# ---- Stage 2: intercorr prefilter ----
prefilter_by_target_k = reduce_correlated_features(
    split_train_df=train_df_k,
    target_cols=target_cols_diag,
    feature_set=kept_k,
    correlation_threshold=intercorr_threshold,
    importance_metric=intercorr_importance_metric,
)

current_features_by_target = {t: list(prefilter_by_target_k.get(t, [])) for t in target_cols_diag}
for t in target_cols_diag:
    if len(current_features_by_target.get(t, [])) == 0:
        current_features_by_target[t] = kept_k[:1]

_print_eliminated('intercorr_prefilter', kept_k, current_features_by_target.get(target_col, []))

# ---- Stage 3: sequential pipeline steps ----
for step in pipeline_steps_diag:
    before_step = list(current_features_by_target.get(target_col, []))

    if step == 'stability':
        union_feats = sorted({f for t in target_cols_diag for f in current_features_by_target.get(t, [])})
        if len(union_feats) == 0:
            union_feats = kept_k[:1]

        _freq_by_target, selected_k = stability_selection_features_cv(
            merged_df=train_df_k,
            target_cols=target_cols_diag,
            n_subsamples=100,
            sample_fraction=0.5,
            model_type='elasticnet',
            l1_ratio=0.7,
            alpha=0.01,
            coef_threshold=0.05,
            candidate_features=union_feats,
            random_state=42,
            verbose=False,
        )

        next_features = {}
        for t in target_cols_diag:
            vals = list(selected_k.get(t, []))
            next_features[t] = vals if len(vals) > 0 else list(current_features_by_target.get(t, []))
        current_features_by_target = next_features

        _print_eliminated(f'pipeline_step_{step}', before_step, current_features_by_target.get(target_col, []))

    elif step == 'correlation':
        # calculate_correlations_and_plot tests numeric columns only
        _corr_df_by_target, significant_by_target = calculate_correlations_and_plot(
            merged_df=train_df_k,
            target_col=target_cols_diag,
            p_threshold=0.05,
            fdr_alpha=0.05,
            use_fdr=True,
            normalize=False,
            make_plots=False,
            correlation_threshold=0,
        )

        sig_features_for_split = {}
        for t in target_cols_diag:
            sig_t = [el[0] for el in significant_by_target.get(t, [])]
            allowed_t = set(current_features_by_target.get(t, []))
            sig_features_for_split[t] = [f for f in sig_t if f in allowed_t]

        _print_eliminated('correlation_step_significance', before_step, sig_features_for_split.get(target_col, []))

        reduced_corr = reduce_correlated_features(
            split_train_df=train_df_k,
            target_cols=target_cols_diag,
            feature_set=sig_features_for_split,
            correlation_threshold=0.8,
            importance_metric='spearman',
        )

        next_features = {}
        for t in target_cols_diag:
            vals = list(reduced_corr.get(t, []))
            next_features[t] = vals if len(vals) > 0 else list(current_features_by_target.get(t, []))
        current_features_by_target = next_features

        _print_eliminated(
            'correlation_step_reduce_correlated',
            sig_features_for_split.get(target_col, []),
            current_features_by_target.get(target_col, []),
        )

    elif step == 'forward_sfs':
        sfs_n_features_to_select = globals().get('sfs_n_features_to_select', int(0.1 * len(train_df_k)))
        sfs_inner_cv = globals().get('sfs_inner_cv', 5)
        sfs_scoring = globals().get('sfs_scoring', 'spearman')

        selected_sfs_k = select_features_forward_sfs(
            split_train_df=train_df_k,
            target_cols=target_cols_diag,
            candidate_features=current_features_by_target,
            n_features_to_select=sfs_n_features_to_select,
            model_type='elasticnet',
            cv=sfs_inner_cv,
            scoring=sfs_scoring,
            random_state=42,
            min_improvement=0.01,
            n_jobs=-1,
        )

        next_features = {}
        for t in target_cols_diag:
            vals = list(selected_sfs_k.get(t, []))
            next_features[t] = vals if len(vals) > 0 else list(current_features_by_target.get(t, []))
        current_features_by_target = next_features

        _print_eliminated(f'pipeline_step_{step}', before_step, current_features_by_target.get(target_col, []))

    else:
        print(f"[diag] Unhandled step: {step}")

print("==== End elimination tracking ====\n")

# Optional sanity check: does the final set include the categorical specials?
final_feats = set(current_features_by_target.get(target_col, []))
for special in ['feature_hc_subtype','feature_lc_subtype']:
    print(f"Final contains {special}? {special in final_feats}")



==== Feature elimination tracking (diagnostic-only) ====
split_idx=0, target_col=target_SEC_Monomer
candidate_features_diag count=125
Special missing in this split: feature_hc_subtype
Special missing in this split: feature_lc_subtype
[diag] low_variance_filter: eliminated 0 features for target=target_SEC_Monomer
[]
[diag] intercorr_prefilter: eliminated 32 features for target=target_SEC_Monomer
['charge_metrics_exposed_net_charge', 'charge_metrics_exposed_net_charge_cdr', 'charge_metrics_protein_pi', 'charge_metrics_sap_pos_charge_score', 'cluster_metrics_aromatic_exposed_cluster_largest_size', 'cluster_metrics_negative_exposed_cluster_avg_rel_asa', 'cluster_metrics_pnc_cdr_vicinity', 'cluster_metrics_positive_exposed_cluster_largest_size', 'density_metrics_avg_hydrophobic_all', 'salt_bridges_metrics_avg_salt_all', 'salt_bridges_metrics_number_of_salt_bridges', 'sequence_motives_n_motif_ArgLys', 'sequence_motives_n_motif_AsnAsp', 'sequence_motives_n_motif_AsnAsp_CDRs', 'sequence_motiv